In [44]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os
import shutil

In [45]:
# Creates folder to save results
filepath = 'LSTM/Tests/1'
script_dir = os.path.dirname('lstm.ipynb')
results_dir = os.path.join(script_dir, filepath + '/')

if not os.path.isdir(results_dir):
    os.makedirs(results_dir)

In [ ]:
# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [47]:
# Set random seed for reproducibility
torch.manual_seed(7)
np.random.seed(7)     

In [48]:
# Load Data
try:
    xy = np.load('./coin_data.npy')
except:
    data =  np.loadtxt('./result.csv',delimiter=',',dtype=np.float32)
    np.save('coin_data', data)

# Indexes inputs
X = xy[:,1:] 

# Turns negative values to 0
X[X < 0] = 0

# Indexes Labels
y = xy[:,0] 

In [49]:
# Define Hyper Parameters
hidden_size = 512
num_classes = 7
num_epochs = 500
batch_size = 8
learning_rate = 0.00005
num_layers = 2

input_size = 751  
sequence_length = 1024   
normalize = "none"   

In [50]:
mapping = {
       0:"1 cent",
       1:"2 cent",
       2:"5 cent",
       3:"20 cent",
       4:"50 cent",
       5:"100 cent",
       6:"200 cent"
}

In [ ]:
# Checks Shapes
print(f'XY Shape: {xy.shape}')
print(f'X Shape: {X.shape}')
print(f'y Shape: {y.shape}')

# Checks for Imbalance
coin_1 = 0
coin_2 = 0
coin_5 = 0
coin_20 = 0
coin_50 = 0
coin_100 = 0
coin_200 = 0
for i in range(xy.shape[0]):
    if xy[i,0] == 0:
        coin_1 += 1
    elif xy[i,0] == 1:
        coin_2 += 1
    elif xy[i,0] == 2:
        coin_5 += 1
    elif xy[i,0] == 3:
        coin_20 += 1
    elif xy[i,0] == 4:
        coin_50 += 1
    elif xy[i,0] == 5:
        coin_100 += 1
    elif xy[i,0] == 6:
        coin_200 += 1
        
print(f'coin 1: {coin_1}')
print(f'coin 2: {coin_2}')
print(f'coin 5: {coin_5}')
print(f'coin 20: {coin_20}')
print(f'coin 50: {coin_50}')
print(f'coin 100: {coin_100}')
print(f'coin 200: {coin_200}')

In [ ]:
# Split data 80-20
# Random State So it is reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

# Graph before applied normalization
plt.figure()
plt.title(f'Example Input Data - Before Normalization - Coin Type - {mapping[y[0]]}')
plt.plot(X_train[0])
plt.xlabel("Sample [Frequency 204 kHz]")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

# Normalize 0-1
sc = MinMaxScaler(feature_range=(0,1))

if normalize == "global":
    # sc.fit_transform on X_train and sc.transform on X_test to avoid data leakage
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

elif normalize == "individual":
    X_train = np.array([sc.fit_transform(sample.reshape(-1, 1)).flatten() for sample in X_train])
    X_test = np.array([sc.fit_transform(sample.reshape(-1, 1)).flatten() for sample in X_test])

# Graph after applied normalization
plt.figure()
plt.title(f'Example Input Data - After Normalization - Coin Type - {mapping[y[0]]}')
plt.plot(X_train[0])
plt.xlabel("Sample [Frequency 204 kHz]")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

In [ ]:
# Check for imbalance on train-test sets

train_coin_1 = 0
train_coin_2 = 0
train_coin_5 = 0
train_coin_20 = 0
train_coin_50 = 0
train_coin_100 = 0
train_coin_200 = 0

for i in range(y_train.shape[0]):
    if y_train[i] == 0:
        train_coin_1 += 1
    elif y_train[i] == 1:
        train_coin_2 += 1
    elif y_train[i] == 2:
        train_coin_5 += 1
    elif y_train[i] == 3:
        train_coin_20 += 1
    elif y_train[i] == 4:
        train_coin_50 += 1
    elif y_train[i] == 5:
        train_coin_100 += 1
    elif y_train[i] == 6:
        train_coin_200 += 1
        
print(f'coin 1: {train_coin_1}')
print(f'coin 2: {train_coin_2}')
print(f'coin 5: {train_coin_5}')
print(f'coin 20: {train_coin_20}')
print(f'coin 50: {train_coin_50}')
print(f'coin 100: {train_coin_100}')
print(f'coin 200: {train_coin_200}')

test_coin_1 = 0
test_coin_2 = 0
test_coin_5 = 0
test_coin_20 = 0
test_coin_50 = 0
test_coin_100 = 0
test_coin_200 = 0

for i in range(y_test.shape[0]):
    if y_test[i] == 0:
        test_coin_1 += 1
    elif y_test[i] == 1:
        test_coin_2 += 1
    elif y_test[i] == 2:
        test_coin_5 += 1
    elif y_test[i] == 3:
        test_coin_20 += 1
    elif y_test[i] == 4:
        test_coin_50 += 1
    elif y_test[i] == 5:
        test_coin_100 += 1
    elif y_test[i] == 6:
        test_coin_200 += 1
        
print(f'coin 1: {test_coin_1}')
print(f'coin 2: {test_coin_2}')
print(f'coin 5: {test_coin_5}')
print(f'coin 20: {test_coin_20}')
print(f'coin 50: {test_coin_50}')
print(f'coin 100: {test_coin_100}')
print(f'coin 200: {test_coin_200}')

In [54]:
# Creates TrainCoinDataSet
class TrainCoinDataSet(Dataset):

    def __init__(self, X_train, y_train):
        # Data Loading
        self.x = torch.from_numpy(X_train.astype(np.float32))
        self.y = torch.from_numpy(y_train).type(torch.LongTensor)
        self.n_samples = y_train.shape[0] # n_samples
        print(self.n_samples)

    def __getitem__(self, index):
        # Allows indexing
        return self.x[index], self.y[index]

    def __len__(self):
        # Allows calling length
        return self.n_samples

# Creates TestCoinDataSet
class TestCoinDataSet(Dataset):

    def __init__(self, X_test, y_test):
        # Data Loading
        self.x = torch.from_numpy(X_test.astype(np.float32))
        self.y = torch.from_numpy(y_test).type(torch.LongTensor)
        self.n_samples = y_test.shape[0] # n_samples
        print(self.n_samples)

    def __getitem__(self, index):
        # Allows indexing
        return self.x[index], self.y[index]

    def __len__(self):
        # Allows calling length
        return self.n_samples

In [ ]:
# Creates Datasets
train_data = TrainCoinDataSet(X_train, y_train)
test_data = TestCoinDataSet(X_test, y_test)

In [56]:
# Creates Dataloaders for train and test datasets
train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# Checks Variable Types, Shapes
examples = iter(train_loader)
samples, labels = next(examples)

print(f'Input shape: {samples.shape} , Labels shape: {labels.shape}')
print(f'Input dtype: {samples[0,0].dtype}')
print(f'Label dtype: {labels[0].dtype}')

In [58]:
# Creates Model
 
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTM, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        # Input -> (batch_size, seq_length, input_size)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.5)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self,x):
        # For first iteration
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)

        out, _ = self.lstm(x, (h0,c0))

        # Dropout on output of LSTM
        out = self.dropout(out)

        # out = batch_size, seq_length, hidden_size
        out = out[:, -1, :]  # [: (all samples in batch), -1 (last time step), : (all features in hidden size)]
        
        out = self.fc(out)
        
        return out

In [59]:
model = LSTM(input_size, hidden_size, num_layers, num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss() # Applies softMax
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
print(model)

In [ ]:
# Training loop

# For storing results
history = {'train_loss': [], 'val_loss': [], 'train_accuracy':[], 'val_accuracy': []}

# For confusion matrix
all_preds = []
all_labels = []

# For plotting
train_loss_steps = []
train_accuracy_steps = []

plot_steps, print_steps = 8,1

running_loss_train_steps = 0.0
n_correct_train_steps = 0
n_samples_train_steps = 0

running_loss_train = 0.0
n_correct_train = 0
n_samples_train = 0

running_loss_val = 0.0
n_correct_val = 0
n_samples_val = 0

for epoch in range(num_epochs):
    
    model.train()
    running_loss_train = 0.0
    n_correct_train = 0
    n_samples_train = 0

    # Training loop
    for i, (inputs, labels) in enumerate(train_loader):
        
        inputs = inputs.reshape(-1, sequence_length, input_size).to(device)
        labels = labels.to(device)
        
        # Forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # For Epoch Graph
        running_loss_train += loss.item()
        _, predictions = torch.max(outputs, 1)
        n_correct_train += (predictions == labels).sum().item()
        n_samples_train += labels.shape[0]

        # For Steps Graph
        running_loss_train_steps += loss.item()
        _, predictions = torch.max(outputs, 1)
        n_correct_train_steps += (predictions == labels).sum().item()
        n_samples_train_steps += labels.shape[0]
        
        # Prints Advance
        if (i+1) % print_steps == 0:
            print(f'epoch {epoch+1} / {num_epochs}, step {i+1}/{len(train_loader)}, loss = {loss.item():.4f}')

        # For Plotting Steps Graph
        if (i+1) % plot_steps == 0:
            train_loss_steps.append(running_loss_train_steps / len(train_loader))
            train_accuracy_steps.append(100 * n_correct_train_steps / n_samples_train_steps)
            running_loss_train_steps = 0.0
            n_correct_train_steps = 0
            n_samples_train_steps = 0


    history['train_loss'].append(running_loss_train / len(train_loader))
    history['train_accuracy'].append(100 * n_correct_train / n_samples_train)

    # Test loop
    running_loss_val = 0.0
    n_correct_val = 0
    n_samples_val = 0

    model.eval()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(test_loader):
            
            inputs = inputs.reshape(-1, sequence_length, input_size).to(device)
            labels = labels.to(device)
            
            # Forward
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Accumulate loss
            running_loss_val += loss.item()
            _, predictions = torch.max(outputs, 1)
            n_correct_val += (predictions == labels).sum().item()
            n_samples_val += labels.shape[0]

            # For confusion matrix
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if (i+1) % print_steps == 0:
                print(f'epoch {epoch+1} / {num_epochs}, step {i+1}/{len(test_loader)}, loss = {loss.item():.4f}')

        history['val_loss'].append(running_loss_val / len(test_loader))
        history['val_accuracy'].append(100 * n_correct_val / n_samples_val)

In [ ]:
# Plots
fig , ax = plt.subplots(1,2,sharex=True)

ax [0].plot(train_loss_steps, label='Train Loss')
ax [0].set_xlabel('Step')
ax [0].set_ylabel('Loss')
ax [0].legend()

ax [1].plot(train_accuracy_steps, label='Train Accuracy')
ax [1].set_xlabel('Step')
ax [1].set_ylabel('Accuracy')
ax [1].legend()

plt.suptitle("Steps Graph")
plt.savefig(results_dir + '/StepGraph.png')
plt.show() 

fig , ax = plt.subplots(1,2,sharex=True)

ax [0].plot(history['train_loss'], label='Train Loss')
ax [0].plot(history['val_loss'], label='Test Loss')
ax [0].set_xlabel('Epoch')
ax [0].set_ylabel('Loss')
ax [0].legend()

ax [1].plot(history['train_accuracy'], label='Train acc')
ax [1].plot(history['val_accuracy'], label='Test acc')
ax [1].set_xlabel('Epoch')
ax [1].set_ylabel('Accuracy')
ax [1].legend()

plt.suptitle("Epoch Graph")
plt.savefig(results_dir + '/EpochGraph.png')
plt.show() 

In [ ]:
# Confusion Matrix
class_names = ['C1','C2','C5','C20','C50','C100','C200']
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
disp.figure_.savefig(results_dir + 'Confusion_Matrix.png')
plt.show()


In [ ]:
# Print Results
test_accuracy_max= max(history["val_accuracy"])
train_accuracy_max=max(history["train_accuracy"])

idx_max_test=history["val_accuracy"].index(test_accuracy_max)
idx_max_train=history["train_accuracy"].index(train_accuracy_max)

print(f"Epochs: {num_epochs} - Trainning accuracy: {history["train_accuracy"][-1]} - Test accuaracy: {history["val_accuracy"][-1]} ")
print(f"Max test accuracy: {max(history["val_accuracy"])} - Epoch: {idx_max_test}")
print(f"Max train accuracy: {max(history["train_accuracy"])} - Epoch: {idx_max_train}")

In [65]:
# Saves Model
torch.save(model.state_dict(), results_dir + "model_weights.pth")

In [ ]:
# Saves Test
shutil.copyfile('./lstm.ipynb', './' + results_dir + 'lstm_copy.ipynb')